# Portfolio Optimization Project: V1 Implementation

This notebook implements Version 1 (V1) of a binary portfolio optimization system, focusing on an objective function and a cardinality constraint. It is designed to be beginner-friendly and mathematically verifiable.

## Section 1: Imports

In [100]:
# Import necessary libraries
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt # Optional, for basic plotting if desired

## Section 2: Load uploaded files

In [101]:
import pandas as pd
from google.colab import files
import io

print("Please upload 'mu.csv'.")
# Upload mu.csv
uploaded_mu = files.upload()
# Get the actual filename Colab used for mu.csv (e.g., 'mu.csv' or 'mu (1).csv')
mu_actual_filename = list(uploaded_mu.keys())[0]
mu_file_content = uploaded_mu[mu_actual_filename].decode('utf-8')
mu = pd.read_csv(io.StringIO(mu_file_content), index_col=0).squeeze() # Squeeze to get a Series/1D array

print("Please upload 'Sigma.csv'.")
# Upload Sigma.csv
uploaded_sigma = files.upload()
# Get the actual filename Colab used for Sigma.csv
sigma_actual_filename = list(uploaded_sigma.keys())[0]
sigma_file_content = uploaded_sigma[sigma_actual_filename].decode('utf-8')
Sigma = pd.read_csv(io.StringIO(sigma_file_content), index_col=0)

print("Files uploaded and loaded successfully.")

Please upload 'mu.csv'.


Saving mu.csv to mu (3).csv
Please upload 'Sigma.csv'.


Saving Sigma.csv to Sigma (3).csv
Files uploaded and loaded successfully.


In [102]:
# Perform sanity checks

# Print mu shape
print(f"mu shape: {mu.shape}")

# Print Sigma shape
print(f"Sigma shape: {Sigma.shape}")

# Check for missing values
if mu.isnull().any() or Sigma.isnull().any().any():
    print("Warning: Missing values detected in mu or Sigma.")
else:
    print("No missing values detected.")

# Expected shapes
expected_mu_shape = (20,)
expected_sigma_shape = (20, 20)

if mu.shape == expected_mu_shape and Sigma.shape == expected_sigma_shape:
    print(f"Shapes match expected: mu {expected_mu_shape}, Sigma {expected_sigma_shape}.")
else:
    print(f"Warning: Shapes do not match expected. Expected mu {expected_mu_shape}, Sigma {expected_sigma_shape}. Actual mu {mu.shape}, Sigma {Sigma.shape}.")

mu shape: (20,)
Sigma shape: (20, 20)
No missing values detected.
Shapes match expected: mu (20,), Sigma (20, 20).


## Section 2.5: Stock and Sector Definitions

In [103]:
stocks = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN", # Technology
    "JPM","V","MA","GS","BAC", # Finance
    "LLY","JNJ","MRK","PFE","ABBV", # Healthcare
    "XOM","CVX","GE","CAT","HON" # Industrial_Energy
]

# Define the sector map based on the 20 stocks
sector_map = {
    "Technology": list(range(0, 5)), # Indices 0-4
    "Finance": list(range(5, 10)),  # Indices 5-9
    "Healthcare": list(range(10, 15)), # Indices 10-14
    "Industrial_Energy": list(range(15, 20)) # Indices 15-19
}

print("Stock list defined.")
print("Sector map defined:", sector_map)

Stock list defined.
Sector map defined: {'Technology': [0, 1, 2, 3, 4], 'Finance': [5, 6, 7, 8, 9], 'Healthcare': [10, 11, 12, 13, 14], 'Industrial_Energy': [15, 16, 17, 18, 19]}


## Section 3: Define frozen parameters

In [104]:
# Define frozen parameters for the optimization problem

# K: Target portfolio size (number of stocks to select)
K = 6
print(f"Target portfolio size (K): {K}")

# lambda_values: Risk-aversion parameters to sweep through
# Lower lambda means more emphasis on expected return, higher lambda emphasizes diversification.
lambda_values = [0.1, 0.5, 1.0, 2.0]
print(f"Lambda values for sweep: {lambda_values}")

# alpha: Cardinality penalty coefficient
# This parameter penalizes portfolios that do not meet the target cardinality K.
alpha = 20
print(f"Cardinality penalty coefficient (alpha): {alpha}")

# Number of stocks (derived from mu length, assuming it's consistent with Sigma)
n = len(mu)
print(f"Total number of available stocks (n): {n}")

Target portfolio size (K): 6
Lambda values for sweep: [0.1, 0.5, 1.0, 2.0]
Cardinality penalty coefficient (alpha): 20
Total number of available stocks (n): 20


In [105]:
# C: Maximum number of stocks allowed per sector
# This is the 'sector_cap' parameter.
sector_cap = 2
print(f"Maximum stocks per sector (sector_cap, C): {sector_cap}")

# beta_values: Sector penalty coefficients to sweep through
# Higher beta means more emphasis on adhering to sector cap.
beta_values = [0, 0.1, 1, 10]
print(f"Beta values for sweep: {beta_values}")

Maximum stocks per sector (sector_cap, C): 2
Beta values for sweep: [0, 0.1, 1, 10]


## Section 4: Define energy/objective function

In [106]:
def energy(
    x,
    mu,
    Sigma,
    lam,
    alpha,
    K,
    sector_map=None,
    sector_cap=2,
    beta=0
):
    mu = np.array(mu)
    Sigma = np.array(Sigma)
    x = np.array(x)

    # Return term
    term1 = -np.dot(mu, x)

    # Risk term
    term2 = lam * np.dot(x.T, np.dot(Sigma, x))

    # Cardinality term
    term3 = alpha * (np.sum(x) - K)**2

    # Sector penalty
    term4 = 0
    if sector_map is not None:
        term4 = sector_penalty(
            x,
            sector_map,
            sector_cap,
            beta
        )

    total_energy = term1 + term2 + term3 + term4

    return total_energy

In [107]:
# Helper function to get sector counts for a given portfolio
def get_sector_counts(portfolio_x, sector_map):
    counts = {}
    for sector_name, indices in sector_map.items():
        counts[sector_name] = np.sum(portfolio_x[indices])
    return counts

# Helper function to check for sector cap violation
def check_sector_violation(sector_counts, sector_cap):
    violations = {sector: count for sector, count in sector_counts.items() if count > sector_cap}
    if violations:
        return f"Violated: {violations}"
    else:
        return "None"

In [108]:
def sector_penalty(x, sector_map, sector_cap, beta):
    """
    Computes the sector exposure penalty for a given portfolio x.

    P_sec(x) = beta * sum_j max(0, sector_count_j - sector_cap)^2

    Args:
        x (np.array): A binary vector representing the portfolio.
        sector_map (dict): A dictionary mapping sector names to lists of stock indices.
        sector_cap (int): The maximum number of stocks allowed per sector.
        beta (float): The sector penalty coefficient.

    Returns:
        float: The computed sector penalty.
    """
    penalty = 0.0
    for sector_name, indices in sector_map.items():
        # Count selected stocks in the current sector
        sector_count_j = np.sum(x[indices])
        # Calculate violation
        violation = max(0, sector_count_j - sector_cap)
        # Add to total penalty
        penalty += beta * (violation**2)
    return penalty

## Section 5: Tiny verification stage

In [109]:
# This tiny verification stage is crucial for ensuring the objective function and basic logic
# are correctly implemented before running on the full dataset. It allows us to manually
# check results for a small, exhaustive search space.

# Take only the first 5 stocks for this small test
num_stocks_small = 5
mu_small = mu[:num_stocks_small]
Sigma_small = Sigma.iloc[:num_stocks_small, :num_stocks_small]

# Set a smaller target portfolio size for the test
K_small = 2

# Use an arbitrary lambda for verification
lam_small = 0.5

print(f"--- Tiny Verification Stage (first {num_stocks_small} stocks, K={K_small}, lambda={lam_small}) ---")
print(f"mu_small shape: {mu_small.shape}")
print(f"Sigma_small shape: {Sigma_small.shape}")

# Generate all binary portfolios for these 5 stocks (2^5 = 32 possibilities)
# Using itertools.product for a brute-force check on a small scale.
all_portfolios_small = list(itertools.product([0, 1], repeat=num_stocks_small))

min_energy_small = float('inf')
best_portfolio_small = None

# Evaluate all 32 possibilities
for portfolio_x in all_portfolios_small:
    current_energy = energy(portfolio_x, mu_small, Sigma_small, lam_small, alpha, K_small)
    if current_energy < min_energy_small:
        min_energy_small = current_energy
        best_portfolio_small = portfolio_x

stocks = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "JPM","V","MA","GS","BAC",
    "LLY","JNJ","MRK","PFE","ABBV",
    "XOM","CVX","GE","CAT","HON"
]

selected_indices_small = [i for i, x_val in enumerate(best_portfolio_small) if x_val == 1]
selected_stocks_names_small = [
    stocks[i] for i in selected_indices_small
] # Assuming generic stock names
cardinality_small = np.sum(best_portfolio_small)

print("\nVerification Results:")
print(f"Best portfolio (binary representation): {best_portfolio_small}")
print(f"Selected stocks (indices): {selected_indices_small}")
print(f"Selected stocks (names): {selected_stocks_names_small}")
print(f"Minimum energy: {min_energy_small:.4f}")
print(f"Cardinality: {cardinality_small}")
print(f"Expected cardinality (K_small): {K_small}")

if cardinality_small == K_small:
    print("Cardinality constraint is satisfied for the best portfolio in verification (good).")
else:
    print("Cardinality constraint is NOT satisfied for the best portfolio in verification (check alpha).")

--- Tiny Verification Stage (first 5 stocks, K=2, lambda=0.5) ---
mu_small shape: (5,)
Sigma_small shape: (5, 5)

Verification Results:
Best portfolio (binary representation): (1, 0, 0, 0, 1)
Selected stocks (indices): [0, 4]
Selected stocks (names): ['AAPL', 'AMZN']
Minimum energy: -0.0055
Cardinality: 2
Expected cardinality (K_small): 2
Cardinality constraint is satisfied for the best portfolio in verification (good).


## Section 6: Full V1 implementation

In [123]:
# For the full V1 implementation, we will NOT brute force all 2^n portfolios.
# Instead, we generate ONLY feasible portfolios satisfying `sum(x) == K`.
# This is done using `itertools.combinations`, which is computationally much smarter
# for sparse binary vectors with a fixed number of ones.

print(f"--- Full V1 Implementation (n={n} stocks, target K={K}) ---")

# Generate all combinations of K stock indices out of n available stocks.
# Each combination represents a feasible portfolio that satisfies sum(x) == K.

# This function will be called repeatedly in the lambda sweep, so we'll define a helper here.
def find_best_portfolio_for_lambda(
    current_lambda,
    current_beta,
    mu,
    Sigma,
    alpha,
    K,
    n,
    sector_map,
    sector_cap,
    stock_names
):
    """
    Finds the best portfolio for a given lambda by enumerating all K-stock combinations.
    """
    min_energy_full = float('inf')
    best_portfolio_indices = None

    # Iterate through all combinations of K stocks out of n total stocks.
    # Each combination represents the indices of stocks selected for the portfolio.
    for indices_tuple in itertools.combinations(range(n), K):
        # Create a binary portfolio vector 'x' from the selected indices.
        # All elements are 0 by default, then set 1 at selected indices.
        x = np.zeros(n, dtype=int)
        for idx in indices_tuple:
            x[idx] = 1

        # Evaluate the energy of the current portfolio
        current_energy = energy(
    x,
    mu,
    Sigma,
    current_lambda,
    alpha,
    K,
    sector_map,
    sector_cap,
    current_beta
)

        # If this portfolio has lower energy, it's our new best.
        if current_energy < min_energy_full:
            min_energy_full = current_energy
            best_portfolio_indices = indices_tuple

    # Convert best_portfolio_indices to a readable list of stock names (e.g., 'Stock_0', 'Stock_1')
    selected_stocks = [
    stock_names[i] for i in best_portfolio_indices
]
    return selected_stocks, min_energy_full

--- Full V1 Implementation (n=20 stocks, target K=6) ---


## Section 7: λ sweep

In [111]:
# Run full V2 implementation across lambda and beta values
# Results stored in pandas DataFrame

results = []

print("--- Starting Lambda + Beta Sweep ---")

beta_values = [0, 0.1, 1, 10]

for current_lambda in lambda_values:
    for current_beta in beta_values:

        print(
            f"\nOptimizing for "
            f"lambda = {current_lambda}, "
            f"beta = {current_beta}..."
        )

        # Find best portfolio
        selected_stocks, min_energy = find_best_portfolio_for_lambda(
            current_lambda=current_lambda,
            current_beta=current_beta,
            mu=mu,
            Sigma=Sigma,
            alpha=alpha,
            K=K,
            n=n,
            sector_map=sector_map,
            sector_cap=2,
            stock_names=stocks
        )

        # Build binary selection vector x
        x = np.zeros(n)

        for idx, stock in enumerate(stocks):
            if stock in selected_stocks:
                x[idx] = 1

        # Sector analysis
        sector_counts = get_sector_counts(x, sector_map)

        sector_violation = check_sector_violation(
            sector_counts,
            sector_cap=2
        )

        # Store results
        results.append({
            'lambda': current_lambda,
            'beta': current_beta,
            'selected_stocks': selected_stocks,
            'energy': min_energy,
            'sector_counts': sector_counts,
            'sector_violation': sector_violation
        })

        # Clean readable output
        print(f"  Best portfolio: {selected_stocks}")
        print(f"  Energy: {min_energy:.6f}")
        print(f"  Sector counts: {sector_counts}")
        print(f"  Sector violation: {sector_violation}")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n--- Lambda + Beta Sweep Results ---")
print(results_df.to_string(index=False))

--- Starting Lambda + Beta Sweep ---

Optimizing for lambda = 0.1, beta = 0...
  Best portfolio: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']
  Energy: -0.015277
  Sector counts: {'Technology': np.float64(2.0), 'Finance': np.float64(2.0), 'Healthcare': np.float64(1.0), 'Industrial_Energy': np.float64(1.0)}
  Sector violation: None

Optimizing for lambda = 0.1, beta = 0.1...
  Best portfolio: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']
  Energy: -0.015277
  Sector counts: {'Technology': np.float64(2.0), 'Finance': np.float64(2.0), 'Healthcare': np.float64(1.0), 'Industrial_Energy': np.float64(1.0)}
  Sector violation: None

Optimizing for lambda = 0.1, beta = 1...
  Best portfolio: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']
  Energy: -0.015277
  Sector counts: {'Technology': np.float64(2.0), 'Finance': np.float64(2.0), 'Healthcare': np.float64(1.0), 'Industrial_Energy': np.float64(1.0)}
  Sector violation: None

Optimizing for lambda = 0.1, beta = 10...
  Best portfolio: ['AAPL', 'AMZN'

## Section 8: Simple interpretation

In [112]:
print("--- Interpretation of Results ---")

# Access the first row (lowest lambda) and last row (highest lambda) for comparison
lowest_lambda_result = results_df.iloc[0]
highest_lambda_result = results_df.iloc[-1]

print(f"1. **Low Lambda (e.g., \u03BB = {lowest_lambda_result['lambda']}):**")
print(f"   At low \u03BB, the objective function places more weight on **maximizing expected return**.")
print(f"   The selected portfolio ({lowest_lambda_result['selected_stocks']}) will likely contain stocks with high individual expected returns, potentially accepting higher risk.")

print(f"\n2. **High Lambda (e.g., \u03BB = {highest_lambda_result['lambda']}):**")
print(f"   At high \u03BB, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.")
print(f"   The selected portfolio ({highest_lambda_result['selected_stocks']}) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.")

print("\nIn general, increasing \u03BB shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.")

--- Interpretation of Results ---
1. **Low Lambda (e.g., λ = 0.1):**
   At low λ, the objective function places more weight on **maximizing expected return**.
   The selected portfolio (['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']) will likely contain stocks with high individual expected returns, potentially accepting higher risk.

2. **High Lambda (e.g., λ = 2.0):**
   At high λ, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.
   The selected portfolio (['AMZN', 'MA', 'GS', 'LLY', 'ABBV', 'HON']) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.

In general, increasing λ shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.


#Conformance Tests

In [113]:
print(mu.shape)
print(Sigma.shape)

(20,)
(20, 20)


In [114]:
print(mu.isna().sum())
print(Sigma.isna().sum().sum())

0
0


# TOY VERIFICATION (5 STOCKS)
#

In [115]:
small_n = 5
small_indices = [0, 5, 10, 15, 18]

In [116]:
mu_small = mu.iloc[small_indices]
Sigma_small = Sigma.iloc[
    small_indices,
    small_indices
]

stocks_small = [stocks[i] for i in small_indices]

Step 2: Freeze small problem

In [117]:
K_small = 2
lam_Small = 0

Step 3: Enumerate ALL possibilities

In [118]:
all_portfolios = list(
    itertools.product([0,1], repeat=small_n)
)

Step 4: Evaluate all portfolios

In [119]:
best_energy = float('inf')
best_portfolio = None

In [120]:
for portfolio in all_portfolios:

    portfolio = np.array(portfolio)

    e = energy(
        portfolio,
        mu_small,
        Sigma_small,
        lam_small,
        alpha,
        K_small
    )

    if e < best_energy:
        best_energy = e
        best_portfolio = portfolio

Step 5: Decode selected stocks

In [121]:
selected_indices = np.where(best_portfolio == 1)[0]

In [122]:
selected_stocks = [
    stocks[i] for i in best_portfolio_indices
]

NameError: name 'best_portfolio_indices' is not defined

Step 6: Print results

In [ ]:
print("TOY VERIFICATION RESULTS")
print("="*40)

print("Selected stocks:")
print(selected_stocks)

print("\nCardinality:")
print(np.sum(best_portfolio))

print("\nEnergy:")
print(best_energy)

In [ ]:
print("Mean Returns (mu):")
for stock, val in zip(stocks_small, mu_small):
    print(f"{stock}: {val:.6f}")

print("\nDiagonal of Covariance Matrix (risk proxy):")
for stock, val in zip(stocks_small, np.diag(Sigma_small)):
    print(f"{stock}: {val:.6f}")

# V2 Sector Constraint Test Ground

In [126]:
# ==========================================
# V2 TEST GROUND — FORCE SECTOR CONSTRAINT
# ==========================================

print("\n--- V2 Sector Constraint Test Ground ---")

# Select intentionally tech-heavy subset
test_indices = [0, 1, 2, 4, 5, 10]
# AAPL, MSFT, NVDA, AMZN, JPM, LLY

test_stocks = [stocks[i] for i in test_indices]

# Reduced mu and Sigma
mu_test = mu.iloc[test_indices].copy()

# Artificially boost tech returns
mu_test.iloc[0] = 0.010  # AAPL
mu_test.iloc[1] = 0.011  # MSFT
mu_test.iloc[2] = 0.012  # NVDA
mu_test.iloc[3] = 0.009  # AMZN

# Lower non-tech attractiveness
mu_test.iloc[4] = 0.001  # JPM
mu_test.iloc[5] = 0.001  # LLY

Sigma_test = Sigma.iloc[
    test_indices,
    test_indices
]

# Local sector map for toy experiment
sector_map_test = {
    "Technology": [0, 1, 2, 3],  # AAPL, MSFT, NVDA, AMZN
    "Finance": [4],              # JPM
    "Healthcare": [5]            # LLY
}

# Freeze parameters
K_test = 4
n_test = len(test_indices)
lambda_test = 0.1
sector_cap_test = 2

print(f"Test stocks: {test_stocks}")
print(f"K = {K_test}")
print(f"Sector cap = {sector_cap_test}")

# Compare beta values
test_beta_values = [0, 10]

for beta_test in test_beta_values:

    selected_stocks, min_energy = find_best_portfolio_for_lambda(
        current_lambda=lambda_test,
        current_beta=beta_test,
        mu=mu_test,
        Sigma=Sigma_test,
        alpha=alpha,
        K=K_test,
        n=n_test,
        sector_map=sector_map_test,
        sector_cap=sector_cap_test,
        stock_names=test_stocks
    )

    # Build x vector
    x_test = np.zeros(n_test)

    for idx, stock in enumerate(test_stocks):
        if stock in selected_stocks:
            x_test[idx] = 1

    # Sector analysis
    sector_counts = get_sector_counts(
        x_test,
        sector_map_test
    )

    violation_status = check_sector_violation(
        sector_counts,
        sector_cap_test
    )

    print("\n" + "="*50)
    print(f"Beta = {beta_test}")
    print("="*50)

    print("Selected Portfolio:")
    print(selected_stocks)

    print("\nEnergy:")
    print(round(min_energy, 6))

    print("\nSector Counts:")
    print(sector_counts)

    print("\nSector Violation:")
    print(violation_status)


--- V2 Sector Constraint Test Ground ---
Test stocks: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'LLY']
K = 4
Sector cap = 2

Beta = 0
Selected Portfolio:
['AAPL', 'MSFT', 'NVDA', 'AMZN']

Energy:
-0.041839

Sector Counts:
{'Technology': np.float64(4.0), 'Finance': np.float64(0.0), 'Healthcare': np.float64(0.0)}

Sector Violation:
Violated: {'Technology': np.float64(4.0)}

Beta = 10
Selected Portfolio:
['MSFT', 'NVDA', 'JPM', 'LLY']

Energy:
-0.024922

Sector Counts:
{'Technology': np.float64(2.0), 'Finance': np.float64(1.0), 'Healthcare': np.float64(1.0)}

Sector Violation:
None
